In [188]:
standard_channel_list = [
'FP1', 
'FP2', 
'F3', 
'F4', 
'C3', 
'C4', 
'P3', 
'P4', 
'O1', 
'O2', 
'F7', 
'F8', 
'T3', 
'T4', 
'T5', 
'T6', 
'CZ']

In [189]:
import os
import h5py
import shutil

def load_data_from_h5(train_path_h5, tmp_path, segment_file_path):
    h5_path = os.path.join(train_path_h5, segment_file_path + '.h5')
    tmp_file_path = os.path.join(tmp_path,  os.path.basename(segment_file_path) + '.h5')
    
    shutil.copyfile(h5_path, tmp_file_path)
    
    # Read the data from the temporary H5 file
    try:
        with h5py.File(tmp_file_path, 'r') as f:
            data = f['eeg'][:]
    except Exception as e:
        data = None
        
    # Clean up the temporary file
    try:
        os.remove(tmp_file_path)
    except Exception as e:
        print(f"Error deleting temporary file: {e}")
        
    return data

train_path_h5 = '/home/students/wcao/eeg_train_seizure_h5'
tmp_path = '/tmp'
segment_file_path = 'aaaaanme/s010_2014/01_tcp_ar/aaaaanme_s010_t012'
data = load_data_from_h5(train_path_h5, tmp_path, segment_file_path)

In [190]:
print(data.shape)

(17, 173250)


In [191]:
from sklearn.preprocessing import StandardScaler

# Transpose to (samples, features) format for StandardScaler
scaler = StandardScaler()
normalized_data = scaler.fit_transform(data.T).T

print(f"Original shape: {data.shape}")
print(f"Normalized shape: {normalized_data.shape}")
print(f"Mean: {normalized_data.mean():.6f}")
print(f"Std: {normalized_data.std():.6f}")

Original shape: (17, 173250)
Normalized shape: (17, 173250)
Mean: 0.000000
Std: 1.000000


In [192]:
# Define EEG frequency bands
frequency_bands = {
    'Delta': (0.5, 4),
    'Theta': (4, 8), 
    'Alpha': (8, 13),
    'Beta': (13, 30),
    'Gamma': (30, 100)
}

sfreq = 250.0  # Sampling frequency in Hz

In [193]:
import numpy as np
from pybispectra import (
    PAC,
    Bispectrum,
    ResultsCFC,
    Threenorm,
    compute_fft,
    get_example_data_paths,
)

In [194]:
n_time_points = normalized_data.shape[1]

n_channels, n_time_points = normalized_data.shape
n_epoch = int(n_time_points // sfreq)


segmented_data = normalized_data.reshape(n_channels, n_epoch, int(sfreq))

segmented_data = segmented_data.transpose(1, 0, 2) # (n_epochs, n_channels, n_time_points_per_epoch)

print(f"Segmented data shape: {segmented_data.shape}")

Segmented data shape: (693, 17, 250)


In [ ]:
fft_coeffs, freqs = compute_fft(
    data=segmented_data, sampling_freq=sfreq, verbose=False
)

print(
    f"FFT coeffs.: [{fft_coeffs.shape[0]} epochs x {fft_coeffs.shape[1]} channels x "
    f"{fft_coeffs.shape[2]} frequencies]\nFreq. range: {freqs[0]} - {freqs[-1]} Hz"
)

for freq in freqs:
    print(f"{freq} Hz", end=", ")

FFT coeffs.: [693 epochs x 17 channels x 126 frequencies]
Freq. range: 0.0 - 125.0 Hz
0.0 Hz, 1.0 Hz, 2.0 Hz, 3.0 Hz, 4.0 Hz, 5.0 Hz, 6.0 Hz, 7.0 Hz, 8.0 Hz, 9.0 Hz, 10.0 Hz, 11.0 Hz, 12.0 Hz, 13.0 Hz, 14.0 Hz, 15.0 Hz, 16.0 Hz, 17.0 Hz, 18.0 Hz, 19.0 Hz, 20.0 Hz, 21.0 Hz, 22.0 Hz, 23.0 Hz, 24.0 Hz, 25.0 Hz, 26.0 Hz, 27.0 Hz, 28.0 Hz, 29.0 Hz, 30.0 Hz, 31.0 Hz, 32.0 Hz, 33.0 Hz, 34.0 Hz, 35.0 Hz, 36.0 Hz, 37.0 Hz, 38.0 Hz, 39.0 Hz, 40.0 Hz, 41.0 Hz, 42.0 Hz, 43.0 Hz, 44.0 Hz, 45.0 Hz, 46.0 Hz, 47.0 Hz, 48.0 Hz, 49.0 Hz, 50.0 Hz, 51.0 Hz, 52.0 Hz, 53.0 Hz, 54.0 Hz, 55.0 Hz, 56.0 Hz, 57.0 Hz, 58.0 Hz, 59.0 Hz, 60.0 Hz, 61.0 Hz, 62.0 Hz, 63.0 Hz, 64.0 Hz, 65.0 Hz, 66.0 Hz, 67.0 Hz, 68.0 Hz, 69.0 Hz, 70.0 Hz, 71.0 Hz, 72.0 Hz, 73.0 Hz, 74.0 Hz, 75.0 Hz, 76.0 Hz, 77.0 Hz, 78.0 Hz, 79.0 Hz, 80.0 Hz, 81.0 Hz, 82.0 Hz, 83.0 Hz, 84.0 Hz, 85.0 Hz, 86.0 Hz, 87.0 Hz, 88.0 Hz, 89.0 Hz, 90.0 Hz, 91.0 Hz, 92.0 Hz, 93.0 Hz, 94.0 Hz, 95.0 Hz, 96.0 Hz, 97.0 Hz, 98.0 Hz, 99.0 Hz, 100.0 Hz, 101.0 Hz, 102.

In [196]:
# compute the bispectrum where kmn = xyy & plot results
bs = Bispectrum(
    data=fft_coeffs, freqs=freqs, sampling_freq=sfreq, verbose=False
)  # initialise object
bs.compute(indices=((1,), (16,), (16,)))  # kmn = xyy
print(bs.results.shape)

(1, 126, 126)


In [197]:
bs_result = np.abs(bs.results.get_results())

bs_result = bs_result.squeeze()
print(bs_result.shape)

(126, 126)


In [198]:
import numpy as np

# 假设 freqs 是从 compute_fft 得到的频率数组，形状 (126,)
# 例如 freqs = np.arange(126)  # 0 to 125 Hz，如果不是，请替换为实际的 freqs

# 定义频带映射函数
def get_band(freq, bands):
    for band, (low, high) in bands.items():
        if low <= freq < high:
            return band
    return None  # 如果不在任何频带

# 假设你的矩阵是 bicoh 或 bs_result，形状 (126, 126)
# 示例：matrix = bicoh.squeeze()  # 或 bs_result

# 初始化字典来存储频带组合
band_combinations = {}

# 遍历矩阵
for i in range(bs_result.shape[0]):
    band1 = get_band(i, frequency_bands)
    if band1 is None:
        continue
    for j in range(bs_result.shape[1]):
        band2 = get_band(j, frequency_bands)
        if band2 is None:
            continue
        value = bs_result[i, j]
        if np.isnan(value):
            continue
        key = (band1, band2)
        if key not in band_combinations:
            band_combinations[key] = []
        band_combinations[key].append(value)

# 计算每个组合的平均值（或其他统计）
for key, values in band_combinations.items():
    median = np.median(values)
    print(f"{key}: Median = {median:.4f}, Count = {len(values)}")

('Delta', 'Delta'): Median = 1086.3712, Count = 6
('Delta', 'Theta'): Median = 366.9133, Count = 12
('Delta', 'Alpha'): Median = 48.2993, Count = 15
('Delta', 'Beta'): Median = 4.0522, Count = 51
('Delta', 'Gamma'): Median = 0.1429, Count = 210
('Theta', 'Theta'): Median = 169.2947, Count = 10
('Theta', 'Alpha'): Median = 38.3614, Count = 20
('Theta', 'Beta'): Median = 1.7662, Count = 68
('Theta', 'Gamma'): Median = 0.0529, Count = 280
('Alpha', 'Alpha'): Median = 2.0803, Count = 15
('Alpha', 'Beta'): Median = 0.2885, Count = 85
('Alpha', 'Gamma'): Median = 0.0226, Count = 350
('Beta', 'Beta'): Median = 0.0583, Count = 153
('Beta', 'Gamma'): Median = 0.0067, Count = 1184
('Gamma', 'Gamma'): Median = 0.0038, Count = 1122


In [199]:
# compute the threenorm
norm = Threenorm(
    data=fft_coeffs, freqs=freqs, sampling_freq=sfreq, verbose=False
)  # initialise object
norm.compute(indices=((0,), (1,), (1,)))  # kmn = xyy

# normalise the bispectrum
bicoh = np.abs(
    bs.results.get_results(copy=False) / norm.results.get_results(copy=False)
)

bicoh = bicoh.squeeze()

band_combinations_bicoh = {}

# 遍历矩阵
for i in range(bicoh.shape[0]):
    band1 = get_band(i, frequency_bands)
    if band1 is None:
        continue
    for j in range(bicoh.shape[1]):
        band2 = get_band(j, frequency_bands)
        if band2 is None:
            continue
        value = bicoh[i, j]
        if np.isnan(value):
            continue
        key = (band1, band2)
        if key not in band_combinations_bicoh:
            band_combinations_bicoh[key] = []
        band_combinations_bicoh[key].append(value)
# 计算每个组合的平均值（或其他统计）
for key, values in band_combinations_bicoh.items():
    median = np.median(values)
    print(f"{key}: Median = {median:.4f}, Count = {len(values)}")

('Delta', 'Delta'): Median = 0.0376, Count = 6
('Delta', 'Theta'): Median = 0.0700, Count = 12
('Delta', 'Alpha'): Median = 0.0267, Count = 15
('Delta', 'Beta'): Median = 0.0137, Count = 51
('Delta', 'Gamma'): Median = 0.0032, Count = 210
('Theta', 'Theta'): Median = 0.0774, Count = 10
('Theta', 'Alpha'): Median = 0.0528, Count = 20
('Theta', 'Beta'): Median = 0.0178, Count = 68
('Theta', 'Gamma'): Median = 0.0020, Count = 280
('Alpha', 'Alpha'): Median = 0.0109, Count = 15
('Alpha', 'Beta'): Median = 0.0064, Count = 85
('Alpha', 'Gamma'): Median = 0.0020, Count = 350
('Beta', 'Beta'): Median = 0.0043, Count = 153
('Beta', 'Gamma'): Median = 0.0020, Count = 1184
('Gamma', 'Gamma'): Median = 0.0037, Count = 1122
